# Naïve Bayes

**Escuela Superior de Cómputo**

Unidad Académica: *Big Data*

Equipo: 
- Ismael Porto García
- Daniel Armas Ramírez

Profesor:
Miguel Sánchez Brito

1. Carga de librerías

In [1]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import sum, count, avg, collect_set
from typing import List, Tuple, Dict
import math

spark = SparkSession.builder.appName("NaiveBayes").getOrCreate()

In [2]:
def transpose(cols: List[str], df: DataFrame, colgroup: str) -> DataFrame:
    """
    Función para rotar el dataframe
    
    Parameters:
        cols(List[str]): Lsita de las columnas del DataFrame
        df(DataFrame): Dataframe al cual se le hará transpose
        colgroup(str): columna usada para agrupar
    
    Returns:
        DataFrame: Una copia del dataframe aplicando Transpose
    """
    rows = []
    columns = df.agg(collect_set(colgroup)).collect()[0][0]
    columns.insert(0, 'temp-col')
    for col in cols:
        if col == colgroup:
            continue
        count_v = df.groupBy(colgroup, col).agg(count("*")).collect()
        for item in count_v:
            temp_row = [f"{col}-{item[col]}"]
            for i in range(1, len(columns)):
                if columns[i] == item[colgroup]:
                    temp_row.append(item['count(1)'])
                else:
                    temp_row.append(0)
            rows.append(temp_row)

    transpose_df = spark.createDataFrame(rows, columns)
    return transpose_df


def read_csv(path:str="naive_data.csv") -> List[str]:
    """
    Función para leer el archivo csv
    Parameters:
        path(str): path de ubicación del archivo csv
    Returns:
        List[DataFrame, List[str]]: retorna el DataFrame y la lista de las columnas
    """
    try:
        df = spark.read.csv(path, inferSchema=True, header=True)
        cols = df.columns

    except Exception as e:
        print(f'Error leyendo el archivo')
        print(str(e))
    else:
        return df, cols

In [4]:
def get_prediction_vect(cols: List[str], df: DataFrame) -> Dict[str, str]:
    """
    Función para construir el vector sobre el cuál se hará la clasificación
    Parameters:
        cols(List[str]): Lista de las columnas del dataframe
        df(DataFrame): Dataframe
    Returns:
        Dict[str,str]: Vector con valores seleccionados de las categorías
    """
    print("Construye el vector de predicción")
    vector = {}

    for col in cols:
        unicos = [row[col] for row in df.select(col).distinct().collect()]

        print(f"Columna {col}: valores únicos -> {unicos}")
        while True:
            option = input(f"Escoge un valor para '{col}' de la lista anterior: ")
            if option in unicos:
                option = option
                break
            else:
                print("Valor no válida. Por favor, selecciona un valor de la lista.")

        if not option == "":
            vector[col] = option

    print("\nVector de predicción construido:")
    return vector

# 1. A Priori

In [5]:
def p_priori(target_col:str , df:DataFrame) -> Dict[str, float]:
    # 1. Definir el conjunto de clases C
    classes = [val[f'{target_col}'] for val in df.select(df[f'{target_col}']).distinct().collect()]

    # 2. Calcular probabilidades a priori para cada uno de las diferentes clases
    a_priori_probs = {}
    n = df.count()
    
    for class_val in classes:
        count = df.filter(df[f'{target_col}'] == class_val).count()
        print(f'Existen {count} registros donde {target_col} = {class_val}')
    
        a_priori_probs[class_val] = count/n

    return a_priori_probs, classes

# 2. Likelihoods

In [6]:
def likelihood(classes, vector_prediccion, df: DataFrame, target_col: str):
    likelihoods = {class_val:[] for class_val in classes}
    for col, val in vector_prediccion.items():
        print(f'#### {col} = {val}')
        df_grouped = df.groupBy(f'{col}', f'{target_col}').count()
    
        # Tomar las filas donde se cumpla COL == VAL
        df_grouped = df_grouped.filter(df_grouped[f'{col}'] == f'{val}')
    
        for class_val in classes:
            # Contar registros por clase
            count_class = df.filter(df[f'{target_col}'] == class_val).count()
    
            # Tomar la fila donde TARGET_COL == CLASS_i
            df_temp = df_grouped.filter(df_grouped[f'{target_col}'] == f'{class_val}')
    
            if df_temp.count() == 0:
                print(f'No hay registros para ({col}={val} | {class_val})')
                continue
    
            # Seleccionar la columna target para obtener el conteo de registros
            count_grouped = df_temp.select('count').collect()[0]['count']
            print(f'Para ({col}={val} | {class_val}) hay {count_grouped} registros')
            # Calcular likelihood
            likelihoods[class_val].append(count_grouped / count_class)

    return likelihoods

In [7]:
def clasificacion(likelihoods) -> None:
    final = {}
    for class_val, probs in likelihoods.items():
        product = math.prod(probs)
        final[class_val] = product
    
    prediccion = max(final, key=final.get)
    print("Categoría con mayor probabilidad:", prediccion)
    print("Probabilidad:", final[prediccion])

## Menú del programa

In [10]:
print("Bienvenido al uso del modelo Naïve Bayes")
path = input("Por favor provee el path de ubicación del archivo csv: ")
df, cols = read_csv(path)
print("Muestra del Dataframe")
df.show()

while True:
    try:
        ans_trans = int(input("Desea hacer transpose al Dataframe: 1. Sí\t2. No: "))
        if ans_trans == 1:
            while True:
                group_col = input("Ingrese una columna para agrupar: ")
                if group_col in cols:
                    transpose_df = transpose(cols, df, group_col)
                    transpose_df.show()
                    break
                else:
                    print("No existe esa columna, ingrese de nuevo")
            break
        if ans_trans == 2:
            break
        else:
            print("Por favor, ingrese 1 para Sí o 2 para No.")
    except ValueError:
        print("Por favor, ingrese un número válido (1 o 2).")


while True:
    target_col = input(f'Selecciona una columna objetivo\n{cols}: ')
    if target_col in cols:
        cols.remove(target_col)
        break
    else:
        print("Columna no válida. Por favor, selecciona una columna de la lista.")

vector_prediccion = get_prediction_vect(cols, df)
print(vector_prediccion)
a_priori_probs, classes = p_priori(target_col, df)
print(a_priori_probs)
likelihoods = likelihood(classes, vector_prediccion, df, target_col)
print(likelihoods)
clasificacion(likelihoods)

Bienvenido al uso del modelo Naïve Bayes


Por favor provee el path de ubicación del archivo csv:  naive_data.csv


Muestra del Dataframe
+--------+----+--------+------+----+
| Outlook|Temp|Humidity|  Wind|Play|
+--------+----+--------+------+----+
|   Sunny| Hot|    High|  Weak|  No|
|   Sunny| Hot|    High|Strong|  No|
|Overcast| Hot|    High|  Weak| Yes|
|    Rain|Mild|    High|  Weak| Yes|
|    Rain|Cool|  Normal|  Weak| Yes|
|    Rain|Cool|  Normal|Strong|  No|
|Overcast|Cool|  Normal|Strong| Yes|
|   Sunny|Mild|    High|  Weak|  No|
|   Sunny|Cool|  Normal|  Weak| Yes|
|    Rain|Mild|  Normal|  Weak| Yes|
+--------+----+--------+------+----+



Desea hacer transpose al Dataframe: 1. Sí	2. No:  1
Ingrese una columna para agrupar:  Humidity


+----------------+------+----+
|        temp-col|Normal|High|
+----------------+------+----+
|   Outlook-Sunny|     1|   0|
|Outlook-Overcast|     1|   0|
|    Outlook-Rain|     0|   1|
|    Outlook-Rain|     3|   0|
|Outlook-Overcast|     0|   1|
|   Outlook-Sunny|     0|   3|
|       Temp-Cool|     4|   0|
|        Temp-Hot|     0|   3|
|       Temp-Mild|     1|   0|
|       Temp-Mild|     0|   2|
|       Wind-Weak|     3|   0|
|     Wind-Strong|     0|   1|
|     Wind-Strong|     2|   0|
|       Wind-Weak|     0|   4|
|        Play-Yes|     4|   0|
|         Play-No|     1|   0|
|        Play-Yes|     0|   2|
|         Play-No|     0|   3|
+----------------+------+----+



...

Columna Humidity: valores únicos -> ['High', 'Normal']


Escoge un valor para 'Humidity' de la lista anterior:  High


Columna Wind: valores únicos -> ['Strong', 'Weak']


Escoge un valor para 'Wind' de la lista anterior:  Weak



Vector de predicción construido:
{'Outlook': 'Sunny', 'Temp': 'Cool', 'Humidity': 'High', 'Wind': 'Weak'}
Existen 4 registros donde Play = No
Existen 6 registros donde Play = Yes
{'No': 0.4, 'Yes': 0.6}
#### Outlook = Sunny
Para (Outlook=Sunny | No) hay 3 registros
Para (Outlook=Sunny | Yes) hay 1 registros
#### Temp = Cool
Para (Temp=Cool | No) hay 1 registros
Para (Temp=Cool | Yes) hay 3 registros
#### Humidity = High
Para (Humidity=High | No) hay 3 registros
Para (Humidity=High | Yes) hay 2 registros
#### Wind = Weak
Para (Wind=Weak | No) hay 2 registros
Para (Wind=Weak | Yes) hay 5 registros
{'No': [0.75, 0.25, 0.75, 0.5], 'Yes': [0.16666666666666666, 0.5, 0.3333333333333333, 0.8333333333333334]}
Categoría con mayor probabilidad: No
Probabilidad: 0.0703125
